# Предзащитное обучение моделей SpectrumAI на 2000 NIST-спектрах

Этап 19 фазы 2 (`DEVELOPMENT_PLAN.md`). Запуск в Kaggle Notebook с T4 GPU.

**Что нужно подготовить заранее:**
1. Загрузить `ml/data/processed/predefense_labeled.parquet` в Kaggle Dataset с именем `spectra-id-predefense`.
2. Прикрепить dataset к этому ноутбуку (правая панель → Add Data).
3. Включить GPU T4 в настройках ноутбука.

Ожидаемое время: 1D-CNN ~2–3 ч, contrastive ~4–6 ч. На обрыв сессии есть промежуточные чекпойнты каждые 5 эпох.

## 1. Зависимости

In [ ]:
!pip install -q rdkit transformers==4.44.* structlog pyyaml

## 2. Клонирование SpectrumAI

In [ ]:
import json
import subprocess
import sys
from pathlib import Path

REPO_DIR = Path('/kaggle/working/SpectrumAI')
REPO_URL = 'https://github.com/<YOUR_USER>/SpectrumAI.git'
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)
for path in (str(REPO_DIR / 'ml'), str(REPO_DIR / 'backend')):
    if path not in sys.path:
        sys.path.insert(0, path)
print('SpectrumAI at', REPO_DIR)

## 3. Загрузка датасета и сплит по InChI Key

In [ ]:
import pandas as pd

from pipelines.training.split import split_by_inchi_key

PARQUET = Path('/kaggle/input/spectra-id-predefense/predefense_labeled.parquet')
df = pd.read_parquet(PARQUET)
print('total:', len(df), 'unique molecules:', df['inchi_key'].nunique())

train_idx, val_idx, test_idx = split_by_inchi_key(
    df['inchi_key'].tolist(), ratios=(0.70, 0.15, 0.15), seed=42
)
print(f'train={len(train_idx)} val={len(val_idx)} test={len(test_idx)}')

## 4. Обучение 1D-CNN

Конфиг: `ml/configs/cnn1d_predefense.yaml`. CLI-скрипт `ml/scripts/train_cnn1d.py` принимает путь к конфигу и устройство.

Если сессия оборвалась — перезапусти ячейку: trainer резюмируется с последнего чекпойнта `/kaggle/working/checkpoints/cnn1d-predefense-0.5.0/last.pt`.

In [ ]:
!cd /kaggle/working/SpectrumAI && python ml/scripts/train_cnn1d.py \
    --config ml/configs/cnn1d_predefense.yaml \
    --device cuda \
    --output-dir /kaggle/working/checkpoints/cnn1d-predefense-0.5.0

## 5. Обучение двухбашенной (contrastive) модели

ChemBERTa загружается из HuggingFace, замораживается. Обучается только проекционная голова + spectrum tower.

In [ ]:
!cd /kaggle/working/SpectrumAI && python ml/scripts/train_contrastive.py \
    --config ml/configs/contrastive_predefense.yaml \
    --device cuda \
    --output-dir /kaggle/working/checkpoints/contrastive-predefense-0.5.0

## 6. Финальная оценка на test

Метрики (глава 11 §11.3): micro/macro F1, precision/recall, top-K retrieval recall (K=1,5,10), mAP. Результат — `/kaggle/working/output/predefense_metrics.json`.

In [ ]:
OUTPUT_DIR = Path('/kaggle/working/output')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CNN_CKPT = Path('/kaggle/working/checkpoints/cnn1d-predefense-0.5.0/best.pt')
CONTRASTIVE_CKPT = Path('/kaggle/working/checkpoints/contrastive-predefense-0.5.0/best.pt')

metrics = {
    'phase': 2,
    'data_source': 'real_nist_2000',
    'dataset_size': len(df),
    'split_sizes': {
        'train': len(train_idx),
        'val': len(val_idx),
        'test': len(test_idx),
    },
    'cnn_checkpoint': str(CNN_CKPT),
    'contrastive_checkpoint': str(CONTRASTIVE_CKPT),
}

# TODO (фаза 2): загрузить чекпойнты, прогнать на test-split, заполнить
# metrics полями micro_f1, macro_f1, precision, recall, mAP, top_1, top_5,
# top_10. Текущая заглушка фиксирует структуру файла. После прогона на
# Kaggle перепиши секцию с реальными значениями (см. pipelines.metrics и
# pipelines.retrieval_metrics).

metrics_path = OUTPUT_DIR / 'predefense_metrics.json'
metrics_path.write_text(json.dumps(metrics, ensure_ascii=False, indent=2))
print('Saved metrics scaffold:', metrics)

## 7. Что скачать

После завершения сессии скачайте в локальный `models/` репозитория:

- `/kaggle/working/checkpoints/cnn1d-predefense-0.5.0/best.pt` → `models/cnn1d-predefense-0.5.0/best.pt`
- `/kaggle/working/checkpoints/contrastive-predefense-0.5.0/best.pt` → `models/contrastive-predefense-0.5.0/best.pt`
- `/kaggle/working/output/predefense_metrics.json` → `ml/experiments/predefense/metrics.json`

Затем продолжите этап 20 (см. `docs/KAGGLE_TRAINING.md`).